# Diagnostic complet — Qwen3.6-27B-FP8 + finegrained-fp8 V4

À exécuter après redémarrage du kernel, avant le pipeline OCR V12.
Les anciens patchs/adaptateurs V1 ne doivent pas être chargés.


In [ ]:
import sys, os, platform, importlib.metadata as md
print("="*100); print("TEST 1 — ENVIRONNEMENT"); print("="*100)
print("Python executable :", sys.executable)
print("Python version :", sys.version)
for p in ["torch","transformers","kernels","accelerate","safetensors","huggingface-hub","triton"]:
    try: print(f"{p:20s}: {md.version(p)}")
    except Exception as e: print(f"{p:20s}: ABSENT ({e})")
import torch
print("CUDA disponible :", torch.cuda.is_available())
print("CUDA torch :", torch.version.cuda)
if torch.cuda.is_available():
    print("GPU :", torch.cuda.get_device_name(0))
    print("Capability :", torch.cuda.get_device_capability(0))
    print("VRAM totale GB :", round(torch.cuda.get_device_properties(0).total_memory/1024**3,2))


In [ ]:
import glob, os
print("="*100); print("TEST 2 — REPERTOIRES FINEGRAINED-FP8"); print("="*100)
ROOT="/domino/edv/modelhub/ModelHub-model-huggingface-kernels-community/finegrained-fp8"
print("ROOT existe :", os.path.exists(ROOT))
paths=glob.glob(ROOT+"/**/build/torch-cuda", recursive=True)
for p in sorted(paths): print(p)


In [ ]:
print("="*100); print("TEST 3 — COMPARAISON MAIN / V4"); print("="*100)
MAIN=ROOT+"/main/build/torch-cuda"; V4=ROOT+"/v4/build/torch-cuda"
for name,path in [("MAIN",MAIN),("V4",V4)]:
    print("\n"+"="*80); print(name, path); print("Existe :",os.path.exists(path))
    if os.path.exists(path):
        for f in sorted(glob.glob(path+"/**/*.py",recursive=True)):
            print(" ",os.path.relpath(f,path))


In [ ]:
import json
print("="*100); print("TEST 4 — METADATA V4"); print("="*100)
candidates=[ROOT+"/v4/build/torch-cuda/metadata.json",ROOT+"/v4/metadata.json"]
found=False
for f in candidates:
    if os.path.exists(f):
        found=True; print("\nFichier :",f)
        with open(f) as fh: print(json.dumps(json.load(fh),indent=2))
if not found: print("Aucun metadata.json trouvé dans les emplacements testés.")


In [ ]:
import sys, importlib
print("="*100); print("TEST 5 — IMPORT DIRECT V4"); print("="*100)
V4=ROOT+"/v4/build/torch-cuda"
for name in list(sys.modules):
    if name=="finegrained_fp8" or name.startswith("finegrained_fp8."): del sys.modules[name]
if V4 in sys.path: sys.path.remove(V4)
sys.path.insert(0,V4); importlib.invalidate_caches()
import finegrained_fp8
print("Module réellement chargé :",finegrained_fp8.__file__)
print("Doit contenir /v4/ :", "/v4/" in finegrained_fp8.__file__)


In [ ]:
print("="*100); print("TEST 6 — API REELLE V4"); print("="*100)
for x in sorted(x for x in dir(finegrained_fp8) if not x.startswith("_")):
    print("[CALLABLE]" if callable(getattr(finegrained_fp8,x)) else "[OBJECT] ",x)


In [ ]:
print("="*100); print("TEST 7 — FONCTIONS ATTENDUES"); print("="*100)
wanted=["fp8_act_quant","matmul","matmul_2d","matmul_batched","matmul_grouped",
"w8a8_fp8_matmul","w8a8_block_fp8_matmul","w8a8_tensor_fp8_matmul",
"w8a8_fp8_matmul_batched","w8a8_block_fp8_matmul_batched","w8a8_tensor_fp8_matmul_batched",
"w8a8_fp8_matmul_grouped","w8a8_block_fp8_matmul_grouped","w8a8_tensor_fp8_matmul_grouped"]
for name in wanted: print(f"{name:40s}", "OK" if hasattr(finegrained_fp8,name) else "ABSENT")


In [ ]:
import inspect
print("="*100); print("TEST 8 — SIGNATURES"); print("="*100)
for name in wanted:
    if hasattr(finegrained_fp8,name):
        try: sig=inspect.signature(getattr(finegrained_fp8,name))
        except Exception: sig="signature non disponible"
        print("\n",name,sig)


In [ ]:
print("="*100); print("TEST 9 — PACKAGE KERNELS"); print("="*100)
import kernels
print("kernels version :",getattr(kernels,"__version__","inconnue"))
print("kernels fichier :",kernels.__file__)
print("LayerRepository :",hasattr(kernels,"LayerRepository"))
print("FuncRepository :",hasattr(kernels,"FuncRepository"))


In [ ]:
print("="*100); print("TEST 10 — TRANSFORMERS"); print("="*100)
import transformers
print("Transformers :",transformers.__version__)
try:
    from transformers import AutoProcessor
    print("AutoProcessor : OK")
except Exception as e: print("AutoProcessor : ERREUR",repr(e))
try:
    from transformers import AutoModelForMultimodalLM
    print("AutoModelForMultimodalLM : OK")
except Exception as e: print("AutoModelForMultimodalLM : ERREUR",repr(e))


In [ ]:
print("="*100); print("TEST 11 — QWEN3.6-27B-FP8"); print("="*100)
MODEL_PATH="/domino/edv/modelhub/ModelHub-model-huggingface-Qwen/Qwen3.6-27B-FP8/main"
print("MODEL_PATH :",MODEL_PATH); print("Existe :",os.path.exists(MODEL_PATH))
if os.path.exists(MODEL_PATH):
    for f in sorted(os.listdir(MODEL_PATH)): print(f)


In [ ]:
from transformers import AutoConfig
print("="*100); print("TEST 12 — CONFIG QWEN"); print("="*100)
try:
    config=AutoConfig.from_pretrained(MODEL_PATH,local_files_only=True,trust_remote_code=True)
    print("Config OK")
    print("model_type :",getattr(config,"model_type",None))
    print("architectures :",getattr(config,"architectures",None))
    print("quantization_config :",getattr(config,"quantization_config",None))
except Exception as e: print("ERREUR CONFIG",type(e).__name__,":",e)


In [ ]:
from transformers import AutoProcessor
print("="*100); print("TEST 13 — PROCESSOR QWEN LOCAL"); print("="*100)
try:
    processor=AutoProcessor.from_pretrained(MODEL_PATH,local_files_only=True,trust_remote_code=True)
    print("Processor : OK"); print("Classe :",processor.__class__.__name__)
except Exception as e: print("PROCESSOR : ERREUR",type(e).__name__,":",e)


In [ ]:
print("="*100); print("TEST 14 — MODULES FP8 EN MEMOIRE"); print("="*100)
for name,module in sorted(sys.modules.items()):
    if "finegrained" in name.lower(): print(name,"\n   ",getattr(module,"__file__",None))


## Test 15 — Chargement réel Qwen3.6-27B-FP8
Exécuter seulement après les tests 1 à 14.


In [ ]:
import time, torch
from transformers import AutoModelForMultimodalLM
print("="*100); print("TEST 15 — CHARGEMENT REEL QWEN3.6-27B-FP8"); print("="*100)
torch.cuda.empty_cache(); t0=time.time()
try:
    model=AutoModelForMultimodalLM.from_pretrained(
        MODEL_PATH,local_files_only=True,trust_remote_code=True,
        device_map="cuda:0",torch_dtype="auto")
    model.eval()
    print("MODELE CHARGE : OK")
    print("Classe :",model.__class__.__name__)
    print("Device :",model.device)
    print("VRAM allouée :",round(torch.cuda.memory_allocated(0)/1024**3,2),"GB")
    print("VRAM réservée :",round(torch.cuda.memory_reserved(0)/1024**3,2),"GB")
    print("Temps :",round(time.time()-t0,1),"secondes")
except Exception as e:
    print("MODELE CHARGE : ECHEC")
    print("Type :",type(e).__name__); print("Message :",str(e))
    import traceback; traceback.print_exc()
